In [1]:
import ROOT as r
import fedrarootlogon 
from matplotlib import pyplot as plt
import matplotlib
import awkward as ak
import uproot
import numpy as np
import sys
sys.path.insert(0, "/home/baronunix/Scripts/")
from Clustering_Cosmici_Frammenti import *
import time
import math 


def g_func(x, p1, p2, p3):
    return p1* np.exp(- (x - p2)**2 / (2*p3**2) )
brick_id = "GSI1"

r.gStyle.SetLineScalePS(1)
r.gStyle.SetOptStat("emr")
PLOTS = 0

Welcome to JupyROOT 6.26/06
Load FEDRA libs


In [2]:
%jsroot on

In [ ]:
file1 = r.TFile("b11_vol.root", "READ")
tracks_V = file1.Get("tracks_n")

In [ ]:
ranges = []
N_volumes = 4 
for i in range(N_volumes):
    ranges.append([])

In [ ]:
def get_first_populated_bin(histo):
    for i in range(histo.GetNbinsX()):
        if (histo.GetBinContent(i)>0):
            return histo.GetXaxis().GetMean(i)

In [ ]:
for iplate in range(31, 67):
    tracks_V.Draw("s.Volume()>>h"+str(iplate)+"(100, 0, 20000)", "s.Plate()=="+str(iplate))
    n_volume = (iplate-31)%4

    h = r.gDirectory.Get("h"+str(iplate))
    low_value = h.GetBinCenter(h.FindFirstBinAbove(0))
    high_value = h.GetBinCenter(h.FindLastBinAbove(0))
    ranges[n_volume].append([low_value, high_value])

In [ ]:
ranges[1]

In [ ]:
reference_intervals = []

for ivolume in range(len(ranges)):
    max_length = 0
    lengths = []
    for iregion in range(len(ranges[ivolume])):
        length = ranges[ivolume][iregion][1] - ranges[ivolume][iregion][0]
        lengths.append(length)
        if (length>max_length):
            max_length = length
    pos = lengths.index(max_length)
    reference_intervals.append(ranges[ivolume][pos])

reference_intervals

Formula per cambiare intervallo 
y = a*x + b, x è tra [c, d], y è tra [e, f]
y(c) = a*c + b = e
y(d) = a*d + b = f 

a*(c-d) = (e-f) -> a = (e-f) / (c-d)
b = e - a*c

In [ ]:
plates_corrections = []
for iplate in range(31, 67):
    ivolume = (iplate-31)%4
    iregion = math.floor((iplate-31)/4)
    e, f = reference_intervals[ivolume][0], reference_intervals[ivolume][1]
    c, d = ranges[ivolume][iregion][0], ranges[ivolume][iregion][1]
    a = (e-f)/(c-d)
    b = e - a*c
    plates_corrections.append([a, b])

In [ ]:
c = r.TCanvas()
c.Divide(3,3)
start_plate = 31

histos = []
for i in range(9):
    hs = r.THStack("hs"+str(i), "")
    histos.append(hs)

for i in range(1, 10):
    temp_hs = []
    for iplate in range(start_plate+(i-1)*4, start_plate+(i)*4):
        if (i>9):
            break
        if ((iplate-start_plate)%4 == 0):
            continue
        a, b = plates_corrections[iplate-start_plate]
        iregion = math.floor((iplate-start_plate)/4)
        ivolume = (iplate-start_plate)%4
        tracks_V.Draw(str(a)+ "*s.Volume() + "+str(b) + ">>hc"+str(iplate)+"(100, 0, 20000)", "s.Plate()=="+str(iplate))
        print(" Drawn Plate " + str(iplate))

        hc = r.gDirectory.Get("hc"+str(iplate))
        hc.SetLineWidth(2)
        if (ivolume==1):
            hc.SetLineColor(r.kRed+2)
        elif(ivolume==2):
            hc.SetLineColor(r.kBlue+2)
        elif(ivolume==3):
            hc.SetLineColor(r.kMagenta+2)
        
        histos[iregion].Add(hc)
        temp_hs.append(hc)
    
    leg = r.TLegend()
    print("len temp_hs ", len(temp_hs))
    leg.AddEntry(temp_hs[0], "VR1")
    leg.AddEntry(temp_hs[1], "VR2")
    leg.AddEntry(temp_hs[2], "VR3")
    c.cd(1)
    histos[i-1].Draw("nostackAP")
    leg.Draw("sames")
    print("\n")

c.Draw()

In [ ]:
iplate = 31 + 3 + 4*3
c = r.TCanvas()
h = r.gDirectory.Get("h"+str(iplate))
hc = r.gDirectory.Get("hc"+str(iplate))
hc.Draw()
h.Draw("same")
c.Draw()

## Creazione Nuovo Tree

In [ ]:
outname = "b11_vol_corr.root"
outFile = r.TFile(outname, "RECREATE")
new_tracks = tracks_V.CloneTree(0)

VR0, VR1, VR2, VR3 = np.zeros(1, dtype=np.float32), np.zeros(1, dtype=np.float32), np.zeros(1, dtype=np.float32), np.zeros(1, dtype=np.float32)
new_tracks.Branch("VR0_avc", VR0, "VR0_avc/F")
new_tracks.Branch("VR1_avc", VR1, "VR1_avc/F")
new_tracks.Branch("VR2_avc", VR2, "VR2_avc/F")
new_tracks.Branch("VR3_avc", VR3, "VR3_avc/F")

for track in tracks_V:
    plate_0 = 31
    vr0, vr1, vr2, vr3 = 0,0,0,0
    k0, k1, k2, k3 = 0,0,0,0
    for s in track.s:
        if((s.Plate()-plate_0)%4 - 0 == 0):
            vr0 += plates_corrections[iplate-start_plate][0]*s.Volume() + plates_corrections[iplate-start_plate][1]
            k0+=1
        elif((s.Plate()-plate_0)%4 - 1 == 0):
            vr1 += plates_corrections[iplate-start_plate][0]*s.Volume() + plates_corrections[iplate-start_plate][1]
            k1+=1
        elif((s.Plate()-plate_0)%4 - 2 == 0):
            vr2 += plates_corrections[iplate-start_plate][0]*s.Volume() + plates_corrections[iplate-start_plate][1]
            k2+=1
        elif((s.Plate()-plate_0)%4 - 3 == 0):
            vr3+= plates_corrections[iplate-start_plate][0]*s.Volume() + plates_corrections[iplate-start_plate][1]
            k3+=1
    if(k0!=0):
        VR0[0] = vr0/k0
    else:
        VR0[0] = 0
    if(k1!=0):
        VR1[0] = vr1/k1
    else:
        VR1[0] = 0
    if(k2!=0):
        VR2[0] = vr2/k2
    else:
        VR2[0] = 0
    if(k3!=0):
        VR3[0] = vr3/k3
    else:
        VR3[0] = 0

    new_tracks.Fill()

outFile.cd()
new_tracks.Write("tracks")
outFile.Close()


## Test PCA 123

In [3]:
outname = "b11_vol_corr.root"
outFile = r.TFile(outname, "READ")
tracks = outFile.Get("tracks")

In [4]:
file_pca = r.TFile("PCA_123.root", "RECREATE")

#campione_pca = cluster_v2.CopyTree("k1 >=1 && k2>=2 && k3>=1 || k1>=1 && k2>=1 && k3>=2")
campione_pca = tracks.CopyTree("VR0_av>6000 && k1>=5 && k2>=5 && k3>=5")
campione_pca.Write("pca")

file_pca.Close()

In [5]:
principal = r.TPrincipal(3, "ND")

file_pca = r.TFile("PCA_123.root", "READ")
info_pca = file_pca.Get("pca")

for track in info_pca:
    vr0, vr1, vr2 = track.VR1_avc, track.VR2_avc, track.VR3_avc
    vrs = np.zeros(3)
    vrs[0] = vr0
    vrs[1] = vr1
    vrs[2] = vr2
    principal.AddRow(vrs)
    
principal.MakePrincipals()
principal.MakeCode()
r.gInterpreter.ProcessLine('.L pca.C+')

0

Writing on file "pca.C" ... done


Info in <TUnixSystem::ACLiC>: creating shared library /home/baronunix/Scripts/GSI1/pca_C.so
Warning in cling::IncrementalParser::CheckABICompatibility():
  Possible C++ standard library mismatch, compiled with __GLIBCXX__ '20220324'
  Extraction of runtime standard library version was: '20220421'


In [6]:
r.gSystem.Load("pca_C.so")

a0, b0 = 2600, 0.6
a1, b1 = 3850, 0.7
a2, b2 = (a1+a0)/2, 0.65

vr123s, vr123b = [], []
vr0s, vr1s, vr2s, vr3s = [], [], [], []
thetas = []
for track in tracks:
    if (track.VR0_av < a2*(1+ np.exp(b2*track.tan*track.tan)) and track.k1<2 and track.k2<2 and track.k3<2): #and track.k1<2 and track.k2<2 and track.k3<2
        continue 
    if (track.k1<1 or track.k2<1 or track.k3<1 or track.k0<1):
        continue
    vr2, vr3, vr0 = track.VR1_avc, track.VR2_avc, track.VR3_avc
    vrs = np.zeros(3)
    vrs[0] = vr2
    vrs[1] = vr3
    vrs[2] = vr0
    princ = np.zeros(3)
    principal.X2P(vrs, princ)
    vr123s.append(princ)
    vr0s.append(track.VR0_avc)
    vr1s.append(track.VR1_avc)
    vr2s.append(track.VR2_avc)
    vr3s.append(track.VR3_avc)
    thetas.append(track.tan)

vr123, vr123b = [], []
for i in vr123s:
    vr123.append(i[0])
    vr123b.append(i[1])
    
file_pca.Close()

In [7]:
file_pca_2 = r.TFile("PCA2.root", "RECREATE")

pca_1 = r.TNtuple("pca_1", "", "VR123:VR123b:VR0_avc:VR1_avc:VR2_avc:VR3_avc:tan")

for i in range(len(vr123)):
    pca_1.Fill(vr123[i], vr123b[i], vr0s[i], vr1s[i], vr2s[i], vr3s[i], thetas[i])

In [8]:
kn = r.TCanvas()
pca_1.Draw("VR123>>123(80, -4., 7.)")

hi = r.gDirectory.Get("123")
hi.SetTitle("VP_{123}; VP_{123}; Entries")
hi.Draw("PE")

legend = r.TLegend(0.6,0.65,0.88,0.85)
legend.SetTextFont(0)
legend.SetTextSize(0.04)
legend.AddEntry("Cut", "Cut: k1 >=1 && k2>=2 && k3>=1 || k1>=1 && k2>=1 && k3>=2", "")
#legend.Draw("SAME")

t1 = r.TText(2, 200, brick_id)
t1.SetTextColor(1)
t1.SetTextSize(20)
t1.Draw("SAME")

kn.Draw()

In [9]:
##for k0>=1
pca_1.Draw("VR123>>v123s(100, -5., 7.)", "", "COLZ")

r.gStyle.SetOptStat(0)
histos = r.gDirectory.Get("v123s")
#histos.SetTitle("VP_{123} [GSI1]; VP_{123}; Entries")
histos.SetTitle("; VP_{123}; Entries")
fit_func = r.TF1("fit_func", "[0]*TMath::Gaus(x, [1], [2]) + [3]*TMath::Gaus(x, [4], [5]) + [6]*TMath::Gaus(x, [7], [8]) + [9]*TMath::Gaus(x, [10], [11])", -4, 7)
fit_func.SetParameters(501, -1.5, 1, 110, 0.9, 0.2, 30, 4.5, 0.5)

fit_func.SetParameter(9, 100)
fit_func.SetParameter(10, 2.)
fit_func.SetParameter(11, 1.5)

#ampiezze
fit_func.SetParLimits(0, 150, 520)
fit_func.SetParLimits(3, 20, 200)
fit_func.SetParLimits(6, 20, 200)

#punto medio
fit_func.SetParLimits(1, -3, -1.)
fit_func.SetParLimits(4, -1., 2.5)
fit_func.SetParLimits(7, 2., 6.)

#deviazione_st
fit_func.SetParLimits(2, 0., 2.5)
fit_func.SetParLimits(5, 0.2, 2.5)
fit_func.SetParLimits(8, 0.3, 2.5)


tr = -4
tr2 = 6
histos.Fit("fit_func", "S", "", tr, tr2)
histos.GetFunction("fit_func").SetLineColor(2)
histos.GetXaxis().SetTitleSize(0.05)

c = r.TCanvas()
histos.Draw("PE")

params = fit_func.GetParameters()

comp1 = r.TF1("comp1", "[0]*TMath::Gaus(x, [1], [2])", tr, 6.)
comp1.SetParameters(params[0], params[1], params[2])
comp1.SetLineColor(4)
comp1.Draw("SAME")

comp2 = r.TF1("comp2", "[0]*TMath::Gaus(x, [1], [2])", -3, 6.)
comp2.SetParameters(params[3], params[4], params[5])
comp2.SetLineColor(95)
comp2.Draw("SAME")

comp3 = r.TF1("comp3", "[0]*TMath::Gaus(x, [1], [2])", -3, 6.)
comp3.SetParameters(params[6], params[7], params[8])
comp3.SetLineColor(8)
comp3.Draw("SAME")

comp4 = r.TF1("comp4", "[0]*TMath::Gaus(x, [1], [2])", -4., tr2)
comp4.SetParameters(params[0], params[1], params[2])
comp4.SetLineColor(4)
comp4.SetLineStyle(2)
comp4.Draw("SAME")

comp5 = r.TF1("comp5", "[0]*TMath::Gaus(x, [1], [2])", -4., tr2)
comp5.SetParameters(params[9], params[10], params[11])
comp5.SetLineColor(r.kCyan+2)
comp5.SetLineStyle(2)
comp5.Draw("SAME")

legend = r.TLegend(0.6,0.55,0.88,0.85)
legend.SetTextFont(0)
legend.SetTextSize(0.04)
#legend.AddEntry(histo, "VR123", "lpe")
legend.AddEntry(fit_func, "Overall Fit")
legend.AddEntry(comp1, "Z = 2")
legend.AddEntry(comp2, "Z = 3")
legend.AddEntry(comp3, "Z > 3")
legend.AddEntry("testo", "Entries = " + str(histos.GetEntries()), "")
legend.AddEntry("chi2 / NDF", "#chi^{2} / NDF = " + str(round(fit_func.GetChisquare(), 2)) + " / " + str(fit_func.GetNDF()), "")
legend.AddEntry("prob", "Prob = " + str(round(fit_func.GetProb(), 10)), "")
legend.Draw("SAME")

line = r.TLine(tr, 0, tr, 320)
line.SetLineColor(12)
line.SetLineStyle(2)
line.SetLineWidth(2)
line.Draw("SAME")

line = r.TLine(tr2, 0, tr2, 320)
line.SetLineColor(12)
line.SetLineStyle(2)
line.SetLineWidth(2)
#line.Draw("SAME")

t1 = r.TText(10000, 270, brick_id)
t1.SetTextColor(1)
t1.SetTextSize(20)
t1.Draw("SAME")

c.Draw()


 FCN=89.0986 FROM MIGRAD    STATUS=CONVERGED    1149 CALLS        1150 TOTAL
                     EDM=4.15589e-07    STRATEGY= 1  ERROR MATRIX UNCERTAINTY   1.2 per cent
  EXT PARAMETER                                   STEP         FIRST   
  NO.   NAME      VALUE            ERROR          SIZE      DERIVATIVE 
   1  p0           2.75809e+02   1.47926e+01  -4.01517e-05  -1.19118e-02
   2  p1          -1.54947e+00   3.28235e-02  -1.45547e-05   1.64072e-02
   3  p2           6.72727e-01   3.24713e-02  -1.97790e-05  -1.32025e-02
   4  p3           1.77650e+02   8.38423e+00   3.31871e-05  -5.51545e-03
   5  p4           1.82637e-01   1.88745e-01   3.04753e-05   1.64093e-03
   6  p5           1.34471e+00   6.82637e-02   6.71968e-05   5.91013e-03
   7  p6           8.57971e+01   7.31691e+00   1.61729e-04   2.30338e-03
   8  p7           4.36558e+00   3.66435e-02   1.39150e-05  -6.76299e-02
   9  p8           3.33694e-01   3.29384e-02  -7.30069e-05   5.63675e-04
  10  p9           1.01664e+0

In [19]:
c = r.TCanvas()
pca_1.Draw("tan>>ht3(100, 0, 0.5)", "VR123>-0.75 && VR123<2.2", "colz") #Z=3?
pca_1.Draw("tan>>ht4(100, 0, 0.5)", "VR123>2.2 && VR123<4.1", "colz") #Z=4?
pca_1.Draw("tan>>ht5(100, 0, 0.5)", "VR123>4.1", "colz") #Z>4?
pca_1.Draw("tan>>ht2(100, 0, 0.5)", "VR123<-0.75", "colz") #Z=2

ht3 = r.gDirectory.Get("ht3")
ht3.SetLineWidth(2)
ht3.SetLineColor(95)

ht4 = r.gDirectory.Get("ht4")
ht4.SetLineWidth(2)
ht4.SetLineColor(r.kCyan+2)

ht5 = r.gDirectory.Get("ht5")
ht5.SetLineWidth(2)
ht5.SetLineColor(8)

ht2 = r.gDirectory.Get("ht2")
ht2.SetLineWidth(2)
ht2.SetLineColor(4)

hs = r.THStack("hts", "")
hs.Add(ht3)
hs.Add(ht4)
hs.Add(ht5)
hs.Add(ht2)
hs.SetTitle(" Angular Distribution Comparison; #theta (rad); Entries ")
hs.Draw("nostack")

leg = r.TLegend()
leg.AddEntry(ht3, "VP_{123}>-0.75 & VP_{123}<2.2")
leg.AddEntry(ht4, "VP_{123}>2.2 & VP_{123}<4.1")
leg.AddEntry(ht5, "VP_{123}>-4.1")
leg.AddEntry(ht2, "VP_{123}<-0.75")
leg.Draw("sames")

c.Draw()

In [39]:
c = r.TCanvas()
pca_1.Draw("VR3_avc>>hv33(80, 0, 15000)", "VR123>-0.75 && VR123<2.2", "colz") #Z=3?
pca_1.Draw("VR3_avc>>hv34(80, 0, 15000)", "VR123>2.2 && VR123<4.1", "colz") #Z=4?
pca_1.Draw("VR3_avc>>hv35(80, 0, 15000)", "VR123>4.1", "colz") #Z>4?
pca_1.Draw("VR3_avc>>hv32(80, 0, 15000)", "VR123<-0.75", "colz") #Z>4?

ht3 = r.gDirectory.Get("hv33")
ht3.SetLineWidth(2)
ht3.SetLineColor(95)

ht4 = r.gDirectory.Get("hv34")
ht4.SetLineWidth(2)
ht4.SetLineColor(r.kCyan+2)

ht5 = r.gDirectory.Get("hv35")
ht5.SetLineWidth(2)
ht5.SetLineColor(8)

ht2 = r.gDirectory.Get("hv32")
ht2.SetLineWidth(2)
ht2.SetLineColor(4)

hs = r.THStack("hts", "")
hs.Add(ht3)
hs.Add(ht4)
hs.Add(ht5)
hs.Add(ht2)
hs.SetTitle(" VR3_avc Comparison; VR3_{av}^{c}; Entries ")
hs.Draw("nostack")

leg = r.TLegend()
leg.AddEntry(ht3, "VP_{123}>-0.75 & VP_{123}<2.2")
leg.AddEntry(ht4, "VP_{123}>2.2 & VP_{123}<4.1")
leg.AddEntry(ht5, "VP_{123}>-4.1")
leg.AddEntry(ht2, "VP_{123}<-0.75")
leg.Draw("sames")

c.Draw()

In [37]:
c = r.TCanvas()
pca_1.Draw("(1/(tan*tan))/1000 + VR123>>h(100, -5, 10)")
c.Draw()

In [13]:
c = r.TCanvas()
pca_1.Draw("VR123>>hv123s(100, -5., 7.)", "VR0_avc<7000", "colz")
c.Draw()

In [14]:
c = r.TCanvas()
pca_1.Draw("VR123:VR0_avc", "", "colz")
c.Draw()

In [15]:
c = r.TCanvas()
pca_1.Draw("tan>>hout(100, 0, 0.5)", "VR0_avc<7000", "colz")
c.Draw()

# Senza Outliers

In [ ]:
r.gSystem.Load("pca_C.so")

a0, b0 = 2600, 0.6
a1, b1 = 3850, 0.7
a2, b2 = (a1+a0)/2, 0.65

vr123s, vr123b = [], []
vr0s, vr1s, vr2s, vr3s = [], [], [], []
thetas = []
for track in tracks:
    if (track.VR0_av < a2*(1+ np.exp(b2*track.tan*track.tan))): #and track.k1<2 and track.k2<2 and track.k3<2
        continue 
    if (track.k1<1 or track.k2<1 or track.k3<1 or track.k0<1):
        continue
    vr2, vr3, vr0 = track.VR1_avc, track.VR2_avc, track.VR3_avc
    vrs = np.zeros(3)
    vrs[0] = vr2
    vrs[1] = vr3
    vrs[2] = vr0
    princ = np.zeros(3)
    principal.X2P(vrs, princ)
    vr123s.append(princ)
    vr0s.append(track.VR0_avc)
    vr1s.append(track.VR1_avc)
    vr2s.append(track.VR2_avc)
    vr3s.append(track.VR3_avc)
    thetas.append(track.tan)

vr123, vr123b = [], []
for i in vr123s:
    vr123.append(i[0])
    vr123b.append(i[1])
    
file_pca.Close()

In [ ]:
file_pca_2 = r.TFile("PCA2.root", "RECREATE")

pca_1 = r.TNtuple("pca_1", "", "VR123:VR123b:VR0_avc:VR1_avc:VR2_avc:VR3_avc:tan")

for i in range(len(vr123)):
    pca_1.Fill(vr123[i], vr123b[i], vr0s[i], vr1s[i], vr2s[i], vr3s[i], thetas[i])

In [ ]:
kn = r.TCanvas()
pca_1.Draw("VR123>>123(80, -4., 7.)")

hi = r.gDirectory.Get("123")
hi.SetTitle("VP_{123}; VP_{123}; Entries")
hi.Draw("PE")

legend = r.TLegend(0.6,0.65,0.88,0.85)
legend.SetTextFont(0)
legend.SetTextSize(0.04)
legend.AddEntry("Cut", "Cut: k1 >=1 && k2>=2 && k3>=1 || k1>=1 && k2>=1 && k3>=2", "")
#legend.Draw("SAME")

t1 = r.TText(2, 200, brick_id)
t1.SetTextColor(1)
t1.SetTextSize(20)
t1.Draw("SAME")

kn.Draw()

In [ ]:
##for k0>=1
pca_1.Draw("VR123>>v123s(100, -5., 7.)", "", "COLZ")

r.gStyle.SetOptStat(0)
histos = r.gDirectory.Get("v123s")
#histos.SetTitle("VP_{123} [GSI1]; VP_{123}; Entries")
histos.SetTitle("; VP_{123}; Entries")
fit_func = r.TF1("fit_func", "[0]*TMath::Gaus(x, [1], [2]) + [3]*TMath::Gaus(x, [4], [5]) + [6]*TMath::Gaus(x, [7], [8]) ", -4, 7)
fit_func.SetParameters(501, -1.5, 1, 110, 0.9, 0.2, 30, 4.5, 0.5)

#ampiezze
fit_func.SetParLimits(0, 150, 520)
fit_func.SetParLimits(3, 20, 200)
fit_func.SetParLimits(6, 20, 200)

#punto medio
fit_func.SetParLimits(1, -3, -1.)
fit_func.SetParLimits(4, -1., 2.5)
fit_func.SetParLimits(7, 2., 6.)

#deviazione_st
fit_func.SetParLimits(2, 0., 2.5)
fit_func.SetParLimits(5, 0.2, 2.5)
fit_func.SetParLimits(8, 0.3, 2.5)


tr = -4
tr2 = 6
histos.Fit("fit_func", "S", "", tr, tr2)
histos.GetFunction("fit_func").SetLineColor(2)
histos.GetXaxis().SetTitleSize(0.05)

c = r.TCanvas()
histos.Draw("PE")

params = fit_func.GetParameters()

comp1 = r.TF1("comp1", "[0]*TMath::Gaus(x, [1], [2])", tr, 6.)
comp1.SetParameters(params[0], params[1], params[2])
comp1.SetLineColor(4)
comp1.Draw("SAME")

comp2 = r.TF1("comp2", "[0]*TMath::Gaus(x, [1], [2])", -3, 6.)
comp2.SetParameters(params[3], params[4], params[5])
comp2.SetLineColor(95)
comp2.Draw("SAME")

comp3 = r.TF1("comp3", "[0]*TMath::Gaus(x, [1], [2])", -3, 6.)
comp3.SetParameters(params[6], params[7], params[8])
comp3.SetLineColor(8)
comp3.Draw("SAME")

comp4 = r.TF1("comp4", "[0]*TMath::Gaus(x, [1], [2])", -4., tr2)
comp4.SetParameters(params[0], params[1], params[2])
comp4.SetLineColor(4)
comp4.SetLineStyle(2)
comp4.Draw("SAME")


legend = r.TLegend(0.6,0.55,0.88,0.85)
legend.SetTextFont(0)
legend.SetTextSize(0.04)
#legend.AddEntry(histo, "VR123", "lpe")
legend.AddEntry(fit_func, "Overall Fit")
legend.AddEntry(comp1, "Z = 2")
legend.AddEntry(comp2, "Z = 3")
legend.AddEntry(comp3, "Z > 3")
legend.AddEntry("testo", "Entries = " + str(histos.GetEntries()), "")
legend.AddEntry("chi2 / NDF", "#chi^{2} / NDF = " + str(round(fit_func.GetChisquare(), 2)) + " / " + str(fit_func.GetNDF()), "")
legend.AddEntry("prob", "Prob = " + str(round(fit_func.GetProb(), 10)), "")
legend.Draw("SAME")

line = r.TLine(tr, 0, tr, 320)
line.SetLineColor(12)
line.SetLineStyle(2)
line.SetLineWidth(2)
line.Draw("SAME")

line = r.TLine(tr2, 0, tr2, 320)
line.SetLineColor(12)
line.SetLineStyle(2)
line.SetLineWidth(2)
line.Draw("SAME")

t1 = r.TText(10000, 270, brick_id)
t1.SetTextColor(1)
t1.SetTextSize(20)
t1.Draw("SAME")

c.Draw()


## Aggiungendo tan

In [ ]:
principal = r.TPrincipal(4, "ND")

file_pca = r.TFile("PCA_123.root", "READ")
info_pca = file_pca.Get("pca")

for track in info_pca:
    vr0, vr1, vr2 = track.VR1_avc, track.VR2_avc, track.VR3_avc
    tan = track.tan
    vrs = np.zeros(4)
    vrs[0] = vr0
    vrs[1] = vr1
    vrs[2] = vr2
    if (tan!=0):
        vrs[3] = 1000*(1/tan)**2
    else:
        vrs[3] = 15000
    principal.AddRow(vrs)
    
principal.MakePrincipals()
principal.MakeCode()

In [ ]:
r.gInterpreter.ProcessLine('.L pca.C+')

In [ ]:
r.gSystem.Load("pca_C.so")

a0, b0 = 2600, 0.6
a1, b1 = 3850, 0.7
a2, b2 = (a1+a0)/2, 0.65

vr123s, vr123b = [], []
vr0s, vr1s, vr2s, vr3s = [], [], [], []
thetas = []
for track in tracks:
    if (track.VR0_av < a2*(1+ np.exp(b2*track.tan*track.tan)) and track.k1<2 and track.k2<2 and track.k3<2): #and track.k1<2 and track.k2<2 and track.k3<2
        continue 
    if (track.k1<1 or track.k2<1 or track.k3<1 or track.k0<1):
        continue
    vr2, vr3, vr0 = track.VR1_avc, track.VR2_avc, track.VR3_avc
    vrs = np.zeros(4)
    vrs[0] = vr2
    vrs[1] = vr3
    vrs[2] = vr0
    vrs[3] = 1000*1/(track.tan*track.tan)
    princ = np.zeros(4)
    principal.X2P(vrs, princ)
    vr123s.append(princ)
    vr0s.append(track.VR0_avc)
    vr1s.append(track.VR1_avc)
    vr2s.append(track.VR2_avc)
    vr3s.append(track.VR3_avc)
    thetas.append(track.tan)

vr123, vr123b = [], []
for i in vr123s:
    vr123.append(i[0])
    vr123b.append(i[1])
    
file_pca.Close()

In [ ]:
file_pca_2 = r.TFile("PCA2.root", "RECREATE")

pca_1 = r.TNtuple("pca_1", "", "VR123:VR123b:VR0_avc:VR1_avc:VR2_avc:VR3_avc:tan")

for i in range(len(vr123)):
    pca_1.Fill(vr123[i], vr123b[i], vr0s[i], vr1s[i], vr2s[i], vr3s[i], thetas[i])

In [ ]:
kn = r.TCanvas()
pca_1.Draw("VR123>>123(80, -4., 7.)")

hi = r.gDirectory.Get("123")
hi.SetTitle("VP_{123}; VP_{123}; Entries")
hi.Draw("PE")

legend = r.TLegend(0.6,0.65,0.88,0.85)
legend.SetTextFont(0)
legend.SetTextSize(0.04)
legend.AddEntry("Cut", "Cut: k1 >=1 && k2>=2 && k3>=1 || k1>=1 && k2>=1 && k3>=2", "")
#legend.Draw("SAME")

t1 = r.TText(2, 200, brick_id)
t1.SetTextColor(1)
t1.SetTextSize(20)
t1.Draw("SAME")

kn.Draw()

In [ ]:
##for k0>=1
pca_1.Draw("VR123>>v123s(100, -5., 7.)", "", "COLZ")

r.gStyle.SetOptStat(0)
histos = r.gDirectory.Get("v123s")
#histos.SetTitle("VP_{123} [GSI1]; VP_{123}; Entries")
histos.SetTitle("; VP_{123}; Entries")
fit_func = r.TF1("fit_func", "[0]*TMath::Gaus(x, [1], [2]) + [3]*TMath::Gaus(x, [4], [5]) + [6]*TMath::Gaus(x, [7], [8]) ", -4, 7)
fit_func.SetParameters(501, -1.5, 1, 110, 0.9, 0.2, 30, 4.5, 0.5)

#ampiezze
fit_func.SetParLimits(0, 150, 520)
fit_func.SetParLimits(3, 20, 200)
fit_func.SetParLimits(6, 20, 200)

#punto medio
fit_func.SetParLimits(1, -3, -1.)
fit_func.SetParLimits(4, -1., 2.5)
fit_func.SetParLimits(7, 2., 6.)

#deviazione_st
fit_func.SetParLimits(2, 0., 2.5)
fit_func.SetParLimits(5, 0.2, 2.5)
fit_func.SetParLimits(8, 0.3, 2.5)


tr = -4
tr2 = 6
histos.Fit("fit_func", "S", "", tr, tr2)
histos.GetFunction("fit_func").SetLineColor(2)
histos.GetXaxis().SetTitleSize(0.05)

c = r.TCanvas()
histos.Draw("PE")

params = fit_func.GetParameters()

comp1 = r.TF1("comp1", "[0]*TMath::Gaus(x, [1], [2])", tr, 6.)
comp1.SetParameters(params[0], params[1], params[2])
comp1.SetLineColor(4)
comp1.Draw("SAME")

comp2 = r.TF1("comp2", "[0]*TMath::Gaus(x, [1], [2])", -3, 6.)
comp2.SetParameters(params[3], params[4], params[5])
comp2.SetLineColor(95)
comp2.Draw("SAME")

comp3 = r.TF1("comp3", "[0]*TMath::Gaus(x, [1], [2])", -3, 6.)
comp3.SetParameters(params[6], params[7], params[8])
comp3.SetLineColor(8)
comp3.Draw("SAME")

comp4 = r.TF1("comp4", "[0]*TMath::Gaus(x, [1], [2])", -4., tr2)
comp4.SetParameters(params[0], params[1], params[2])
comp4.SetLineColor(4)
comp4.SetLineStyle(2)
comp4.Draw("SAME")


legend = r.TLegend(0.6,0.55,0.88,0.85)
legend.SetTextFont(0)
legend.SetTextSize(0.04)
#legend.AddEntry(histo, "VR123", "lpe")
legend.AddEntry(fit_func, "Overall Fit")
legend.AddEntry(comp1, "Z = 2")
legend.AddEntry(comp2, "Z = 3")
legend.AddEntry(comp3, "Z > 3")
legend.AddEntry("testo", "Entries = " + str(histos.GetEntries()), "")
legend.AddEntry("chi2 / NDF", "#chi^{2} / NDF = " + str(round(fit_func.GetChisquare(), 2)) + " / " + str(fit_func.GetNDF()), "")
legend.AddEntry("prob", "Prob = " + str(round(fit_func.GetProb(), 10)), "")
legend.Draw("SAME")

line = r.TLine(tr, 0, tr, 320)
line.SetLineColor(12)
line.SetLineStyle(2)
line.SetLineWidth(2)
line.Draw("SAME")

line = r.TLine(tr2, 0, tr2, 320)
line.SetLineColor(12)
line.SetLineStyle(2)
line.SetLineWidth(2)
line.Draw("SAME")

t1 = r.TText(10000, 270, brick_id)
t1.SetTextColor(1)
t1.SetTextSize(20)
t1.Draw("SAME")

c.Draw()
